# Derive CHOMP Eq.(25) from Eq.(23)+(24)

reference **$\color{yellow}{CHOMP: Covariant Hamiltonian Optimization for Motion Planning}$**

## 0. Original formulas (paper)

**Obstacle functional (Eq.23)** — integral over time and body points:

$$
F_{\mathrm{obs}}[\xi]
= \int_{0}^{1}\int_{u\in B}
c\big(x(\xi(t), u)\big)\,
\Big\|
\frac{d}{dt}x(\xi(t), u)
\Big\|\,du\,dt
$$

**Functional gradient / Euler–Lagrange (Eq.24)** — for integrand $v$:

$$
\bar\nabla F
= \frac{\partial v}{\partial \xi}
- \frac{d}{dt}\frac{\partial v}{\partial \xi'}
$$

**Target result (Eq.25)**:

$$
\bar\nabla F_{\mathrm{obs}}[\xi]
= \int_{u\in B}
J^{T}\Big(
\|x'\|\big((I-\hat x'\hat x'^{T})\nabla c - c\,\kappa\big)
\Big)\,du
$$

with curvature (Eq.26)

$$
\kappa
= \frac{1}{\|x'\|^{2}}(I-\hat x'\hat x'^{T})x'' .
$$

## 1. Swap integrals: fix body point $u$ first

Under standard integrability (Fubini), we may swap order:

$$
F_{\mathrm{obs}}[\xi]
= \int_{u\in B}
\underbrace{
\left(
\int_{0}^{1}
c\big(x(\xi(t), u)\big)\,
\Big\|
\frac{d}{dt}x(\xi(t), u)
\Big\|\,dt
\right)
}_{F_u[\xi]\,:\,\text{cost of one fixed body point}}
\,du .
$$

So for a **known / fixed** body point $u\in B$, write the workspace path

$$
x(t) := x(\xi(t), u)
$$

and only study

$$
F_u[\xi]
= \int_{0}^{1} c\big(x(t)\big)\,\|x'(t)\|\,dt .
$$

Afterwards: $\bar\nabla F_{\mathrm{obs}} = \displaystyle\int_{u\in B}\bar\nabla F_u\,du$ (pull back by $J^{T}$).

**Roadmap**

1. Fixed $u$: $L = c(x)\|x'\|$
2. Workspace EL: $\delta F_u/\delta x = \partial L/\partial x - d/dt(\partial L/\partial x')$
3. Rewrite as $\|x'\|[P\nabla c - c\kappa]$
4. $\delta q$: multiply $J^{T}$, then $\int_B$ → Eq.(25)

In [12]:
from sympy import Function, Matrix, Symbol, diff, simplify, sqrt, eye, zeros, latex
from IPython.display import display, Math

def show(title, expr):
    display(Math(rf"=== {title} ==="))
    display(Math(latex(expr)))

In [13]:
# SymPy setup (2D) — used later to check algebra
t = Symbol("t", real=True)
x1, x2 = Function("x_1"), Function("x_2")
c = Function("c")

X = Matrix([x1(t), x2(t)])
Xp = diff(X, t)                 # x'
speed = sqrt(Xp.dot(Xp))        # ||x'||
L = c(x1(t), x2(t)) * speed

show(r"L = c\|x'\|", L)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 2. Workspace Euler–Lagrange for fixed $u$

Integrand $L(x,x')=c(x)\,\|x'\|$. By Eq.(24) in workspace coordinates:

$$
\frac{\delta F_u}{\delta x}
= \frac{\partial L}{\partial x}
- \frac{d}{dt}\frac{\partial L}{\partial x'}.
$$

**Partials (hand)**

$$
\frac{\partial L}{\partial x}=\|x'\|\,\nabla c,
\qquad
\frac{\partial L}{\partial x'}=c\,\hat x'
\quad\text{with}\quad
\hat x'=\frac{x'}{\|x'\|}.
$$

**Expand the time derivative**

$$
\frac{d}{dt}(c\,\hat x')
= \dot c\,\hat x' + c\,\frac{d\hat x'}{dt}.
$$

Chain rule: $\dot c=\nabla c\cdot x'=\|x'\|(\nabla c\cdot\hat x')$, so

$$
\dot c\,\hat x'
=\|x'\|(\hat x'\hat x'^{T})\nabla c.
$$

Hence

\begin{align*}
\frac{\delta F_u}{\delta x}
&= \|x'\|\,\nabla c
- \|x'\|(\hat x'\hat x'^{T})\nabla c
- c\,\frac{d\hat x'}{dt} \\
&= \|x'\|\,(I-\hat x'\hat x'^{T})\nabla c
- c\,\frac{d\hat x'}{dt}.
\end{align*}


## 3. Identify curvature $\kappa$

Differentiate the unit tangent:

$$
\frac{d\hat x'}{dt}
=\frac{d}{dt}\!\left(\frac{x'}{\|x'\|}\right)
=\frac{1}{\|x'\|}(I-\hat x'\hat x'^{T})x''.
$$

Paper definition (Eq.26):

$$
\kappa
=\frac{1}{\|x'\|^{2}}(I-\hat x'\hat x'^{T})x''
\quad\Rightarrow\quad
\frac{d\hat x'}{dt}=\|x'\|\,\kappa.
$$

Substitute back:

$$
\frac{\delta F_u}{\delta x}
=\|x'\|\,(I-\hat x'\hat x'^{T})\nabla c
- c\,\|x'\|\,\kappa
=\|x'\|\;\Big[(I-\hat x'\hat x'^{T})\nabla c - c\,\kappa\Big].
$$

Define projection $P:=I-\hat x'\hat x'^{T}$ (kills the tangential component → retiming-invariant push).


In [15]:
# Step 2a — partials of L
dL_dX = Matrix([diff(L, x1(t)), diff(L, x2(t))])
show(r"\partial L/\partial x = \|x'\|\,\nabla c", simplify(dL_dX))

dL_dXp = Matrix([diff(L, Xp[0]), diff(L, Xp[1])])
show(r"\partial L/\partial x' = c\,\hat x'", simplify(dL_dXp))

# Step 2b+3 — EL residual, then rewrite with P and kappa
EL = simplify(dL_dX - diff(dL_dXp, t))
show(r"\delta F_u/\delta x = \partial L/\partial x - d/dt(\partial L/\partial x')", EL)

s = speed
xhat = Xp / s
P = eye(2) - xhat * xhat.T
Xpp = diff(Xp, t)
kappa = (1 / s**2) * (P * Xpp)
grad_c = Matrix([diff(c(x1(t), x2(t)), x1(t)), diff(c(x1(t), x2(t)), x2(t))])

target = simplify(s * (P * grad_c - c(x1(t), x2(t)) * kappa))
show(r"\|x'\|[(I-\hat x'\hat x'^T)\nabla c - c\kappa]", target)
show(r"EL - target  (should be 0)", simplify(EL - target))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4. Pull back to configuration: $J^{T}$, then $\int_B$

Workspace path depends on configuration: $x=x(q,u)$, $q=\xi(t)$.

$$
\delta x = J\,\delta q,
\qquad
J=\frac{\partial x}{\partial q}(q,u).
$$

Directional change of $F_u$:

$$
\delta F_u
=\int_0^1
\Big\langle\frac{\delta F_u}{\delta x},\,\delta x\Big\rangle dt
=\int_0^1
\Big\langle
J^{T}\frac{\delta F_u}{\delta x},\,
\delta q
\Big\rangle dt.
$$

So the configuration-space functional gradient of one body point is

$$
\bar\nabla_q F_u
=J^{T}\frac{\delta F_u}{\delta x}
=J^{T}\Big(
\|x'\|\;\big[(I-\hat x'\hat x'^{T})\nabla c-c\,\kappa\big]
\Big).
$$

Integrate over body $B$ (linearity of $\bar\nabla$):

$$
\bar\nabla F_{\mathrm{obs}}[\xi]
=\int_{u\in B}
J^{T}\Big(
\|x'\|\;\big[(I-\hat x'\hat x'^{T})\nabla c-c\,\kappa\big]
\Big)\,du
\quad\text{(Eq.25)}.
$$

Done.


## Checklist

| step | claim | status |
|------|--------|--------|
| 1 | Fubini → fix $u$, study $F_u=\int c\|x'\|dt$ | done |
| 2 | $\partial L/\partial x=\|x'\|\nabla c$, $\partial L/\partial x'=c\hat x'$ | sympy |
| 3 | EL $= \|x'\|[P\nabla c-c\kappa]$, $\kappa=\|x'\|^{-2}Px''$ | sympy residual $=0$ |
| 4 | chain rule $\times J^{T}$, $\int_B$ → Eq.(25) | analytic |

**Intuition:** $P\nabla c$ pushes only **sideways** (shape), not along the path (timing). $-c\kappa$ is the curvature / elastic-band term from differentiating $\|x'\|$.
